# Siphuncle Builder

Build a simplified siphuncle running through the chambered part of the shell.

Conceptually, the siphuncle is treated as a small tube threading through the internal chamber septa.

In a chambered cephalopod shell, the siphuncle does not need to extend all the way to the open mouth of the shell. Instead, it should run through the chambered region and reach the final chamber wall near the start of the living body chamber.

To keep the model simple and visually coherent, the siphuncle path is generated from aperture rings along the shell growth path:

1. aperture ring centres are estimated from the shell mesh
2. the siphuncle position is offset slightly from each centre
3. selected positions are used to form a tube path
4. neighbouring tube rings are stitched into a continuous mesh

If chamber_spacing is supplied, siphuncle rings are placed at the same approximate interval as the chamber septa. This keeps the siphuncle visually aligned with the chambered structure rather than stopping too early or running arbitrarily through the body chamber.

# A Note on Aperture Points

The shell surface is stored as a sequence of aperture rings, flattened into
one long vertex array:

    ring 0: vertices 0 .. aperture_points-1
    ring 1: vertices aperture_points .. 2*aperture_points-1
    ring 2: ...

The chamber/septum builder reconstructs each aperture ring by slicing this
vertex array using aperture_points. If this value does not match the value
used to build the shell mesh, the slices no longer correspond to real
aperture rings. They cut across neighbouring rings instead.

Symptoms of a mismatch:
- septa appear as thin discs rather than bowls
- early chambers have odd spacing or size
- siphuncle/chamber geometry may drift or twist
- spiral shells may partially hide the problem, while orthocones expose it

Therefore, dependent internal structures should normally inherit:

    aperture_points = shell_options["aperture_points"]

rather than defining their own independent value in the preset JSON.

# A Note on Aperture Angular Direction

A number of the functions in this notebook have a "fixed_offset_angle" parameter.

If supplied, this is a fixed angular direction (radians) used when offsetting the siphuncle within the aperture cross-section.

If None, the offset direction follows the shell's growth phase, causing the siphuncle to rotate around the shell axis as growth progresses. This is appropriate for many coiled shells.

If a numeric angle is supplied (e.g. 0.0), the offset direction remains constant throughout growth, producing a straight,
consistently positioned siphuncle. This is particularly useful for orthocones and other non-coiled shell forms.

In [1]:
import numpy as np

In [2]:
def unpack_shell_vertices(shell_mesh):
    """
    Extract shell vertex coordinate arrays from a shell mesh.

    :param shell_mesh: Tuple returned by build_shell_mesh
    :return: X, Y, Z coordinate arrays
    """
    X, Y, Z, _, _, _, _, _ = shell_mesh
    return X, Y, Z

In [3]:
def get_shell_ring(X, Y, Z, ring_index, aperture_points):
    """
    Extract one aperture ring from the shell mesh.

    :param X: Shell mesh x coordinates
    :param Y: Shell mesh y coordinates
    :param Z: Shell mesh z coordinates
    :param ring_index: Growth-path ring index
    :param aperture_points: Number of vertices in each aperture ring
    :return: ring_x, ring_y, ring_z arrays
    """
    start = ring_index * aperture_points
    end = start + aperture_points

    return X[start:end], Y[start:end], Z[start:end]

In [4]:
def ring_centre_3d(ring_x, ring_y, ring_z):
    """
    Estimate the 3D centre of an aperture ring.

    :param ring_x: Ring x coordinates
    :param ring_y: Ring y coordinates
    :param ring_z: Ring z coordinates
    :return: centre_x, centre_y, centre_z
    """
    return np.mean(ring_x), np.mean(ring_y), np.mean(ring_z)

In [5]:
def ring_radius_xy(ring_x, ring_y, centre_x, centre_y):
    """
    Estimate the mean aperture radius in the x/y coiling plane for a planispiral/coiled shell

    :param ring_x: Ring x coordinates
    :param ring_y: Ring y coordinates
    :param centre_x: Ring centre x coordinate
    :param centre_y: Ring centre y coordinate
    :return: Mean radial distance in the x/y plane
    """
    return np.mean(
        np.sqrt(
            (ring_x - centre_x)**2 +
            (ring_y - centre_y)**2
        )
    )

In [6]:
def ring_radius_yz(ring_y, ring_z, centre_y, centre_z):
    """
    Estimate the mean aperture radius in the y/z cross-section plane for an orthocone

    This is useful for orthocones, where growth runs along the x axis and the shell aperture lies
    approximately in the y/z plane

    :param ring_x: Ring x coordinates
    :param ring_z: Ring z coordinates
    :param centre_x: Ring centre x coordinate
    :param centre_z: Ring centre z coordinate
    :return: Mean radial distance in the x/z plane
    """
    return np.mean(
        np.sqrt(
            (ring_y - centre_y)**2 +
            (ring_z - centre_z)**2
        )
    )

In [7]:
def max_siphuncle_growth_index(n_spiral, siphuncle_end_fraction):
    """
    Compute the final growth index reached by the siphuncle.

    :param n_spiral: Number of shell growth stages
    :param siphuncle_end_fraction: Fraction of shell growth where siphuncle stops
    :return: Integer growth-path index
    """
    max_index = int((n_spiral - 1) * siphuncle_end_fraction)
    return max(1, min(max_index, n_spiral - 1))

In [8]:
def choose_siphuncle_indices(
    theta,
    chamber_spacing=None,
    siphuncle_end_fraction=0.98,
    include_terminal_ring=True
):
    """
    Choose growth indices where siphuncle tube rings will be placed.

    :param theta: Spiral angle values representing shell growth
    :param chamber_spacing: Optional angular spacing matching chamber septa
    :param siphuncle_end_fraction: Fraction of shell growth where siphuncle stops
    :param include_terminal_ring: Whether to force a final terminal ring
    :return: List of selected growth indices
    """
    n_spiral = len(theta)
    max_index = max_siphuncle_growth_index(n_spiral, siphuncle_end_fraction)

    if chamber_spacing is None:
        return list(range(max_index + 1))

    siphuncle_indices = [0]
    last_siphuncle_theta = theta[0]

    for s in range(1, max_index + 1):
        if theta[s] - last_siphuncle_theta >= chamber_spacing:
            siphuncle_indices.append(s)
            last_siphuncle_theta = theta[s]

    if include_terminal_ring and siphuncle_indices[-1] != max_index:
        siphuncle_indices.append(max_index)

    return siphuncle_indices

In [ ]:
def siphuncle_centre_for_ring(
    theta_value,
    ring_x,
    ring_y,
    ring_z,
    siphuncle_offset,
    fixed_offset_angle
):
    """
    Compute the offset siphuncle centre for one shell aperture ring.

    :param theta_value: Spiral angle at the current growth point
    :param ring_x: Aperture ring x coordinates
    :param ring_y: Aperture ring y coordinates
    :param ring_z: Aperture ring z coordinates
    :param siphuncle_offset: Fraction of available shell radius used for offset
    :param fixed_offset_angle: Siphuncle aperture angular direction
    :param growth_mode: Shell growth mode
    :return: sx, sy, sz siphuncle centre coordinates
    """
    centre_x, centre_y, centre_z = ring_centre_3d(ring_x, ring_y, ring_z)

    radius = ring_radius_xy(ring_x, ring_y, centre_x, centre_y)

    sx = centre_x + siphuncle_offset * radius * np.cos(theta_value)
    sy = centre_y + siphuncle_offset * radius * np.sin(theta_value)
    sz = centre_z

    if fixed_offset_angle is None:
        offset_angle = theta_value
    else:
        offset_angle = fixed_offset_angle

    sx = centre_x + siphuncle_offset * radius * np.cos(offset_angle)
    sy = centre_y + siphuncle_offset * radius * np.sin(offset_angle)
    sz = centre_z

    return sx, sy, sz

In [10]:
def siphuncle_centre_for_orthocone_ring(
    ring_x,
    ring_y,
    ring_z,
    siphuncle_offset,
    siphuncle_radius,
    fixed_offset_angle=0.0
):
    """
    Compute the centre position of an orthocone siphuncle tube ring.

    Orthocone shells grow along a predominantly straight axis, so the
    siphuncle is positioned within the aperture cross-section rather
    than following a coiling path. The offset is applied within the
    local y/z aperture plane while ensuring the siphuncle remains
    inside the shell boundary.

    :param ring_x: Aperture ring x coordinates
    :param ring_y: Aperture ring y coordinates
    :param ring_z: Aperture ring z coordinates
    :param siphuncle_offset: Fraction of available shell radius used for offset
    :param siphuncle_radius: Radius of the siphuncle tube
    :param fixed_offset_angle: Siphuncle aperture angular direction
    :return: Siphuncle centre coordinates (sx, sy, sz)
    """
    centre_x, centre_y, centre_z = ring_centre_3d(ring_x, ring_y, ring_z)

    shell_radius_y = np.max(np.abs(ring_y - centre_y))
    shell_radius_z = np.max(np.abs(ring_z - centre_z))
    shell_radius = max(shell_radius_y, shell_radius_z)

    available_offset = max(shell_radius - siphuncle_radius, 0.0)
    offset_distance = siphuncle_offset * available_offset

    sx = centre_x
    sy = centre_y + offset_distance * np.cos(fixed_offset_angle)
    sz = centre_z + offset_distance * np.sin(fixed_offset_angle)

    return sx, sy, sz

In [11]:
def build_siphuncle_ring(sx, sy, sz, siphuncle_radius, tube_points):
    """
    Build one circular siphuncle tube ring.

    :param sx: Siphuncle centre x coordinate
    :param sy: Siphuncle centre y coordinate
    :param sz: Siphuncle centre z coordinate
    :param siphuncle_radius: Radius of the siphuncle tube
    :param tube_points: Number of vertices around the tube ring
    :return: Xs, Ys, Zs lists for one tube ring
    """
    phi = np.linspace(0, 2*np.pi, tube_points, endpoint=False)

    Xs, Ys, Zs = [], [], []

    for p in phi:
        local_x = siphuncle_radius * np.cos(p)
        local_z = siphuncle_radius * np.sin(p)

        Xs.append(sx + local_x)
        Ys.append(sy)
        Zs.append(sz + local_z)

    return Xs, Ys, Zs

In [12]:
def build_orthocone_siphuncle_ring(sx, sy, sz, siphuncle_radius, tube_points):
    """
    Build one circular siphuncle tube ring for an orthocone.

    The orthocone grows along the x axis, so the siphuncle tube also
    runs along x. Each tube cross-section therefore lies in the y/z plane.
    """
    phi = np.linspace(0, 2*np.pi, tube_points, endpoint=False)

    Xs, Ys, Zs = [], [], []

    for p in phi:
        local_y = siphuncle_radius * np.cos(p)
        local_z = siphuncle_radius * np.sin(p)

        Xs.append(sx)
        Ys.append(sy + local_y)
        Zs.append(sz + local_z)

    return Xs, Ys, Zs

In [ ]:
def shell_radius_for_siphuncle_ring(ring_x, ring_y, ring_z, growth_mode):
    """
    Estimate the local shell radius used when positioning or scaling a siphuncle tube.

    Different shell growth modes use different geometric frames. Orthocones measure
    shell size in the y/z aperture plane, whereas coiled shells use the x/y coiling plane.

    :param ring_x: Aperture ring x coordinates
    :param ring_y: Aperture ring y coordinates
    :param ring_z: Aperture ring z coordinates
    :param growth_mode: Shell growth mode identifier
    :return: Estimated local shell radius
    """
    centre_x, centre_y, centre_z = ring_centre_3d(ring_x, ring_y, ring_z)

    if growth_mode == "orthocone":
        shell_radius_y = np.max(np.abs(ring_y - centre_y))
        shell_radius_z = np.max(np.abs(ring_z - centre_z))
        return max(shell_radius_y, shell_radius_z)

    return ring_radius_xy(ring_x, ring_y, centre_x, centre_y)

In [ ]:
def tapered_siphuncle_radius(
    base_radius,
    shell_radius,
    reference_shell_radius,
    taper_enabled=False,
    taper_power=1.0,
    min_radius=None,
    max_radius=None
):
    """
    Compute the siphuncle radius for a particular growth position.

    When tapering is enabled, the siphuncle radius is scaled relative to the local
    shell radius. This allows the siphuncle to grow or shrink proportionally with
    shell size while optionally enforcing minimum and maximum limits.

    :param base_radius: Reference siphuncle radius at the reference shell size
    :param shell_radius: Local shell radius at the current growth position
    :param reference_shell_radius: Shell radius used as the scaling reference
    :param taper_enabled: Whether siphuncle tapering should be applied
    :param taper_power: Exponent controlling taper strength
    :param min_radius: Optional minimum permitted siphuncle radius
    :param max_radius: Optional maximum permitted siphuncle radius
    :return: Scaled siphuncle radius
    """
    if not taper_enabled or reference_shell_radius <= 0:
        radius = base_radius
    else:
        scale = shell_radius / reference_shell_radius
        radius = base_radius * (scale ** taper_power)

    if min_radius is not None:
        radius = max(radius, min_radius)

    if max_radius is not None:
        radius = min(radius, max_radius)

    return radius

In [15]:
def stitch_tube_rings(
    Is,
    Js,
    Ks,
    face_growth_index,
    siphuncle_indices,
    tube_points
):
    """
    Stitch neighbouring siphuncle tube rings into triangular faces.

    :param Is: First vertex index for each triangle
    :param Js: Second vertex index for each triangle
    :param Ks: Third vertex index for each triangle
    :param face_growth_index: Per-triangle growth sample index along the log spiral
    :param siphuncle_indices: Shell growth indices used to place siphuncle rings
    :param tube_points: Number of vertices around each siphuncle tube ring
    """
    n_rings = len(siphuncle_indices)

    for s in range(n_rings - 1):
        growth_index = siphuncle_indices[s + 1]

        for p in range(tube_points):
            current = s * tube_points + p
            next_p = s * tube_points + ((p + 1) % tube_points)

            current_next = (s + 1) * tube_points + p
            next_next = (s + 1) * tube_points + ((p + 1) % tube_points)

            Is.append(current)
            Js.append(current_next)
            Ks.append(next_p)
            face_growth_index.append(growth_index)

            Is.append(next_p)
            Js.append(current_next)
            Ks.append(next_next)
            face_growth_index.append(growth_index)

In [ ]:
def build_siphuncle_mesh(
    theta,
    shell_mesh,
    aperture_points=32,
    siphuncle_radius=0.6,
    siphuncle_offset=0.35,
    tube_points=16,
    chamber_spacing=None,
    siphuncle_end_fraction=0.98,
    siphuncle_taper_enabled=False,
    siphuncle_taper_power=1.0,
    siphuncle_min_radius=None,
    siphuncle_max_radius=None,
    include_terminal_ring=True,
    fixed_offset_angle=None,
    growth_mode="log_spiral"
):
    """
    Build a simplified siphuncle tube through the chambered shell region.

    :param theta: Spiral angle values representing shell growth
    :param shell_mesh: Mesh returned by build_shell_mesh
    :param aperture_points: Number of vertices around each aperture ring
    :param siphuncle_radius: Radius of the siphuncle tube
    :param siphuncle_offset: Fraction of available shell radius used for offset
    :param tube_points: Number of vertices around the siphuncle tube
    :param chamber_spacing: Optional angular spacing matching chamber septa
    :param siphuncle_end_fraction: Fraction of shell growth where siphuncle stops
    :param siphuncle_taper_enabled: Whether or not to taper the siphuncle
    :param siphuncle_taper_power: Lower values make the siphuncle taper more
    :param siphuncle_min_radius: Minimum siphuncle radius
    :param siphuncle_max_radius: Maximum siphuncle radius
    :param include_terminal_ring: Whether to add a final terminal ring
    :param fixed_offset_angle: Siphuncle aperture angular direction
    :param growth_mode: Shell growth mode
    :return: Xs, Ys, Zs vertex arrays and triangle index arrays Is, Js, Ks, face_growth_index
    """
    X, Y, Z = unpack_shell_vertices(shell_mesh)

    siphuncle_indices = choose_siphuncle_indices(
        theta,
        chamber_spacing=chamber_spacing,
        siphuncle_end_fraction=siphuncle_end_fraction,
        include_terminal_ring=include_terminal_ring
    )

    Xs, Ys, Zs, face_growth_index = [], [], [], []

    shell_radii = []
    for s in siphuncle_indices:
        ring_x, ring_y, ring_z = get_shell_ring(
            X, Y, Z,
            ring_index=s,
            aperture_points=aperture_points
        )

        shell_radii.append(
            shell_radius_for_siphuncle_ring(
                ring_x,
                ring_y,
                ring_z,
                growth_mode=growth_mode
            )
        )

    reference_shell_radius = shell_radii[-1] if shell_radii else 0.0

    for ring_number, s in enumerate(siphuncle_indices):
        current_siphuncle_radius = tapered_siphuncle_radius(
            siphuncle_radius,
            shell_radii[ring_number],
            reference_shell_radius,
            taper_enabled=siphuncle_taper_enabled,
            taper_power=siphuncle_taper_power,
            min_radius=siphuncle_min_radius,
            max_radius=siphuncle_max_radius
        )

        ring_x, ring_y, ring_z = get_shell_ring(
            X, Y, Z,
            ring_index=s,
            aperture_points=aperture_points
        )

        if growth_mode == "orthocone":
            sx, sy, sz = siphuncle_centre_for_orthocone_ring(
                ring_x,
                ring_y,
                ring_z,
                siphuncle_offset=siphuncle_offset,
                siphuncle_radius=current_siphuncle_radius,
                fixed_offset_angle=0.0
            )

            rx, ry, rz = build_orthocone_siphuncle_ring(
                sx,
                sy,
                sz,
                siphuncle_radius=current_siphuncle_radius,
                tube_points=tube_points
            )
        else:
            sx, sy, sz = siphuncle_centre_for_ring(
                theta[s],
                ring_x,
                ring_y,
                ring_z,
                siphuncle_offset=siphuncle_offset,
                fixed_offset_angle=fixed_offset_angle
            )

            rx, ry, rz = build_siphuncle_ring(
                sx,
                sy,
                sz,
                siphuncle_radius=current_siphuncle_radius,
                tube_points=tube_points
            )

        Xs.extend(rx)
        Ys.extend(ry)
        Zs.extend(rz)

    Xs = np.array(Xs)
    Ys = np.array(Ys)
    Zs = np.array(Zs)

    Is, Js, Ks = [], [], []

    stitch_tube_rings(
        Is,
        Js,
        Ks,
        face_growth_index,
        siphuncle_indices=siphuncle_indices,
        tube_points=tube_points
    )

    return Xs, Ys, Zs, Is, Js, Ks, np.array(face_growth_index)